In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [4]:
%%writefile vector_add.cu
#include <iostream>
using namespace std;


__global__ void add(int* A, int* B, int* C, int size) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < size) {
        C[tid] = A[tid] + B[tid];
    }
}

int main() {
    int N;
    cout << "Enter size of vectors: ";
    cin >> N;

    int *A = new int[N];
    int *B = new int[N];
    int *C = new int[N];

    cout << "Enter elements of Vector A:\n";
    for (int i = 0; i < N; i++) {
        cin >> A[i];
    }

    cout << "Enter elements of Vector B:\n";
    for (int i = 0; i < N; i++) {
        cin >> B[i];
    }

    size_t bytes = N * sizeof(int);

    int *d_A, *d_B, *d_C;

    cudaMalloc(&d_A, bytes);
    cudaMalloc(&d_B, bytes);
    cudaMalloc(&d_C, bytes);

    cudaMemcpy(d_A, A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, bytes, cudaMemcpyHostToDevice);

    int threads = 256;
    int blocks = (N + threads - 1) / threads;

    add<<<blocks, threads>>>(d_A, d_B, d_C, N);

    cudaMemcpy(C, d_C, bytes, cudaMemcpyDeviceToHost);

    cout << "Result (A + B):\n";
    for (int i = 0; i < N; i++) {
        cout << C[i] << " ";
    }
    cout << endl;

    delete[] A;
    delete[] B;
    delete[] C;

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Overwriting vector_add.cu


In [5]:
!nvcc vector_add.cu -o vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [6]:
!./vector_add

Enter size of vectors: 5
Enter elements of Vector A:
1 2 3 4 5
Enter elements of Vector B:
5 4 3 2 1
Result (A + B):
6 6 6 6 6 
